# Ⅰ第5回 演習1 学習済みの ResNet18 を自分の写真に合わせる

**問い**：大量の画像（ImageNet：1000 種類・約 128 万枚）で学習済みのモデルを使うと，自分の少ない写真でも正解率は上がるか．

- 第4回 演習1 の自分のセル（`course1/day4/ex1.ipynb`）を **コピーして貼り**，下の表の所だけ直す
- コードパターンを理解するため，**LLM（ChatGPT・Copilot など）を使わずに** やってみよう
- 各テンプレートの下の `check(...)` で `OK` が出たら次へ進む．止まったら，出たメッセージの「ヒント」を読む．それでも分からなければ，テンプレートの上の「ヒント（ほぼ答え）」を開く
- 先に **凍結**（最後の層だけ学習）で①〜⑫を通し，そのあと⑦〜⑫を **微調整**（全部の層を学習）で書き直す

## (0) 番号の設定

In [ ]:
# ===== (0) 番号の設定 =====
MEMBER_NO = 1                 # ← グループ内の自分の番号 (1〜5)．presenter はこの番号の順に演習ごとに交代する

# 授業用フォルダ（AI_TD）のルートを import パスに追加する（ノートブックをどこから開いても動く）
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".dlcourse_root").exists())
sys.path.insert(0, str(ROOT))
print("作業フォルダ:", ROOT)

## (1) 自分のデータ

第4回と同じ `assets/mydata/` を使う（無ければケーブル画像で進む）．

In [ ]:
from common.mydata import image_dir
from common.stepcheck import check
IMAGE_DIR = str(image_dir())     # ③ではこのフォルダを読む
for d in sorted(Path(IMAGE_DIR).iterdir()):
    if d.is_dir():
        print(f"{d.name:20s} {len(list(d.glob('*.jpg')))} 枚")

## 第4回 演習1 から変える所

| テンプレート | 変える所 |
|---|---|
| ① | `from torchvision import models` を足す（学習済みモデルを読むため） |
| ② | `Resize((224, 224))` にし，`ToTensor()` のあとに `transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])` を足す（ImageNet で学習したときと同じ入力にする） |
| ⑦ | 下の「新しく使う書き方」 |
| ⑧⑨ | 凍結は `optim.Adam(model.fc.parameters(), lr=0.001)`，微調整は `optim.Adam(model.parameters(), lr=0.0001)` |
| ⑩ | 凍結は `EPOCHS = 20`，微調整は `EPOCHS = 10` |
| ほか | 同じでよい |

## 新しく使う書き方（⑦）

**凍結**（最後の層だけ学習する）

```python
NUM_CLASSES = len(full_dataset.classes)                    # クラス数
from torchvision.models import ResNet18_Weights
model = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)   # 学習済みの ResNet18 を読む（初回はダウンロード 45MB）
for param in model.parameters():                           # 全パラメータを固定する（勾配を計算しない）
    param.requires_grad = False
num_ftrs = model.fc.in_features                            # 最終層に入る数（512）
model.fc = nn.Linear(num_ftrs, NUM_CLASSES)                # 最終層（1000 クラス用）をクラス数に置き換える
for param in model.fc.parameters():                        # 置き換えた最終層だけを学習対象にする
    param.requires_grad = True
model = model.to(device)
dummy_input = torch.randn(1, 3, 224, 224).to(device)       # ダミーデータで出力の形を確かめる
output = model(dummy_input)
print("出力の形状: ", output.size())                        # torch.Size([1, クラス数]) になる
```

**微調整**（全部の層を学習する）：上の `for` を 2 つとも書かない．⑧⑨は `optim.Adam(model.parameters(), lr=0.0001)` にし，学習率を 1/10 にする．

## 予測（実行する前に書く）

このセルをダブルクリックして書き込む．

- 第4回の自分の CNN のテスト正解率（シード 0）：＿＿％
- ResNet18 凍結：＿＿％　微調整：＿＿％

## データ準備フェーズ（①〜④）

### コードテンプレート①：ライブラリのインポート

<details><summary>ヒント（分からないときに開く・ほぼ答え）</summary>

```python
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from torch.utils.data import Dataset
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import random
import numpy as np
from torchvision.datasets import ImageFolder
from torchvision import models
```

</details>

In [ ]:
check("①", globals(), size=224)

### コードテンプレート②：データ前処理

<details><summary>ヒント（分からないときに開く・ほぼ答え）</summary>

```python
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
}
```

</details>

In [ ]:
check("②", globals(), size=224)

### コードテンプレート③：データの読み込みと分割（乱数の種の固定）

<details><summary>ヒント（分からないときに開く・ほぼ答え）</summary>

```python
full_dataset = ImageFolder(root=IMAGE_DIR)
random.seed(0)
np.random.seed(0)
torch.manual_seed(0)
if torch.backends.mps.is_available():
    torch.mps.manual_seed(0)
train_size = int(0.7 * len(full_dataset))
val_size = int(0.15 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size
train_subset, val_subset, test_subset = random_split(full_dataset, [train_size, val_size, test_size])
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
    def __getitem__(self, index):
        image, label = self.subset[index]
        if self.transform:
            image = self.transform(image)
        return image, label
    def __len__(self):
        return len(self.subset)
train_dataset = CustomDataset(train_subset, transform=data_transforms['train'])
val_dataset = CustomDataset(val_subset, transform=data_transforms['val'])
test_dataset = CustomDataset(test_subset, transform=data_transforms['val'])
```

</details>

In [ ]:
check("③", globals(), size=224)

### コードテンプレート④：データローダの作成

<details><summary>ヒント（分からないときに開く・ほぼ答え）</summary>

```python
BATCH_SIZE = 8
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
```

</details>

In [ ]:
check("④", globals(), size=224)

## 学習設定フェーズ（⑤〜⑨）

### コードテンプレート⑤：使用デバイスの選択

<details><summary>ヒント（分からないときに開く・ほぼ答え）</summary>

```python
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"使用デバイス: {device}")
```

</details>

In [ ]:
check("⑤", globals(), size=224)

### コードテンプレート⑥：ミニバッチの形状確認

<details><summary>ヒント（分からないときに開く・ほぼ答え）</summary>

```python
imgs, _ = next(iter(train_loader))
c, h, w = imgs[0].shape
print(imgs.shape)
```

</details>

In [ ]:
check("⑥", globals(), size=224)

### コードテンプレート⑦：ネットワークの定義（ResNet18・凍結）

<details><summary>ヒント（分からないときに開く・ほぼ答え）</summary>

```python
NUM_CLASSES = len(full_dataset.classes)
from torchvision.models import ResNet18_Weights
model = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
for param in model.parameters():
    param.requires_grad = False
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, NUM_CLASSES)
for param in model.fc.parameters():
    param.requires_grad = True
model = model.to(device)
dummy_input = torch.randn(1, 3, 224, 224).to(device)
output = model(dummy_input)
print("出力の形状: ", output.size())
```

</details>

In [ ]:
check("⑦", globals(), size=224)

### コードテンプレート⑧⑨：損失関数と最適化手法

<details><summary>ヒント（分からないときに開く・ほぼ答え）</summary>

```python
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)
```

</details>

In [ ]:
check("⑧⑨", globals(), size=224)

## 学習・評価フェーズ（⑩〜⑫）

### コードテンプレート⑩：学習

<details><summary>ヒント（分からないときに開く・ほぼ答え）</summary>

```python
EPOCHS = 20
train_loss_list = []
val_loss_list = []
train_acc_list = []
val_acc_list = []
import time
_t0 = time.perf_counter()
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()
    train_loss_list.append(train_loss / train_total)
    train_acc_list.append(100.0 * train_correct / train_total)
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
    val_loss_list.append(val_loss / val_total)
    val_acc_list.append(100.0 * val_correct / val_total)
train_sec = time.perf_counter() - _t0
```

</details>

In [ ]:
check("⑩", globals(), size=224)

### コードテンプレート⑪：学習結果の可視化

<details><summary>ヒント（分からないときに開く・ほぼ答え）</summary>

```python
epochs = range(1, len(train_loss_list) + 1)
plt.figure(figsize=(10, 4))
sns.lineplot(x=epochs, y=train_acc_list, label='Train Accuracy')
sns.lineplot(x=epochs, y=val_acc_list, label='Val Accuracy')
plt.show()
```

</details>

### コードテンプレート⑫：テストデータでの評価

<details><summary>ヒント（分からないときに開く・ほぼ答え）</summary>

```python
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
print(f'Accuracy: {correct / total * 100:.2f}% ')
```

</details>

In [ ]:
check("⑫", globals(), size=224)

## 記録（凍結）

In [ ]:
# 記録（凍結のあとと微調整のあとに 1 回ずつ実行する）
from common.logger import ResultLogger
n_all = sum(p.numel() for p in model.parameters())
n_trained = sum(p.numel() for p in model.parameters() if p.requires_grad)
mode = "freeze" if n_trained < n_all else "finetune"
metrics = {"test_accuracy": correct / total, "val_acc_best": max(val_acc_list) / 100,
           "val_acc_last": val_acc_list[-1] / 100, "train_sec": train_sec, "params_trained": n_trained,
           "n_class": len(full_dataset.classes), "n_images": len(full_dataset), "n_test": total}
logger = ResultLogger(None, f"member{MEMBER_NO}", course="c1", day="d5", exercise="ex1", device=device)
logger.log_many(metrics, condition=f"{Path(IMAGE_DIR).name}_ResNet18_{mode}", seed=0)
print(f"ResNet18（{'凍結' if mode == 'freeze' else '微調整'}）：テスト正解率 {correct / total * 100:.1f}%（{total} 枚中 {correct} 枚）"
      f"｜学習したパラメータ {n_trained:,} / {n_all:,}｜{train_sec:.0f} 秒 → {logger.path.name}")

## 微調整に替える

⑦を「凍結の `for` 2 つなし」で書き直し，⑧⑨は全部のパラメータを `lr=0.0001` で渡す．⑩〜⑫は上の自分のセルをコピーして貼り，`EPOCHS = 10` にする．

### コードテンプレート⑦：ネットワークの定義（ResNet18・微調整）

<details><summary>ヒント（分からないときに開く・ほぼ答え）</summary>

```python
NUM_CLASSES = len(full_dataset.classes)
from torchvision.models import ResNet18_Weights
model = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, NUM_CLASSES)
model = model.to(device)
dummy_input = torch.randn(1, 3, 224, 224).to(device)
output = model(dummy_input)
print("出力の形状: ", output.size())
```

</details>

In [ ]:
check("⑦", globals(), size=224)

### コードテンプレート⑧⑨：損失関数と最適化手法

<details><summary>ヒント（分からないときに開く・ほぼ答え）</summary>

```python
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)
```

</details>

In [ ]:
check("⑧⑨", globals(), size=224)

### コードテンプレート⑩：学習

上の⑩をコピーして貼り，`EPOCHS = 10` にする．

<details><summary>ヒント（分からないときに開く・ほぼ答え）</summary>

```python
EPOCHS = 10
train_loss_list = []
val_loss_list = []
train_acc_list = []
val_acc_list = []
import time
_t0 = time.perf_counter()
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()
    train_loss_list.append(train_loss / train_total)
    train_acc_list.append(100.0 * train_correct / train_total)
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
    val_loss_list.append(val_loss / val_total)
    val_acc_list.append(100.0 * val_correct / val_total)
train_sec = time.perf_counter() - _t0
```

</details>

In [ ]:
check("⑩", globals(), size=224)

### コードテンプレート⑪：学習結果の可視化

上の⑪をコピーして貼る．

<details><summary>ヒント（分からないときに開く・ほぼ答え）</summary>

```python
epochs = range(1, len(train_loss_list) + 1)
plt.figure(figsize=(10, 4))
sns.lineplot(x=epochs, y=train_acc_list, label='Train Accuracy')
sns.lineplot(x=epochs, y=val_acc_list, label='Val Accuracy')
plt.show()
```

</details>

### コードテンプレート⑫：テストデータでの評価

上の⑫をコピーして貼る．

<details><summary>ヒント（分からないときに開く・ほぼ答え）</summary>

```python
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
print(f'Accuracy: {correct / total * 100:.2f}% ')
```

</details>

In [ ]:
check("⑫", globals(), size=224)

## 記録（微調整）

In [ ]:
# 記録（凍結のあとと微調整のあとに 1 回ずつ実行する）
from common.logger import ResultLogger
n_all = sum(p.numel() for p in model.parameters())
n_trained = sum(p.numel() for p in model.parameters() if p.requires_grad)
mode = "freeze" if n_trained < n_all else "finetune"
metrics = {"test_accuracy": correct / total, "val_acc_best": max(val_acc_list) / 100,
           "val_acc_last": val_acc_list[-1] / 100, "train_sec": train_sec, "params_trained": n_trained,
           "n_class": len(full_dataset.classes), "n_images": len(full_dataset), "n_test": total}
logger = ResultLogger(None, f"member{MEMBER_NO}", course="c1", day="d5", exercise="ex1", device=device)
logger.log_many(metrics, condition=f"{Path(IMAGE_DIR).name}_ResNet18_{mode}", seed=0)
print(f"ResNet18（{'凍結' if mode == 'freeze' else '微調整'}）：テスト正解率 {correct / total * 100:.1f}%（{total} 枚中 {correct} 枚）"
      f"｜学習したパラメータ {n_trained:,} / {n_all:,}｜{train_sec:.0f} 秒 → {logger.path.name}")

## 振り返り（提出）

このセルをダブルクリックして，＿＿ に書き込む（それぞれ 1〜2 文）．

1. 第4回の CNN・凍結・微調整のテスト正解率と学習時間（1 回ずつ）：＿＿
2. 凍結で学習したのは最終層の 1,026 個だけ（全体の約 1 万分の 1）．それでも正解率が高かったのはなぜか．最終層の手前までの層が何をしているかに注目して書く：＿＿
3. `check(...)` で `OK` が出なかった（エラーになった）テンプレートの番号と，どう直したか（無ければ「なし」）：＿＿